# 00 - Données, géodésie et orthophotos

Ce notebook contrôle la configuration du projet, explicite les CRS utilisés et prépare le manifeste STAC SWISSIMAGE pour les cinq agglomérations.


In [1]:
# Paramètres
from pathlib import Path

PROJECT_ROOT = Path.cwd()
AGGLO_KEYS = ["lausanne", "berne", "geneve", "bale", "zurich"]
BOUNDARY_SOURCE = "vaco"
ORTHOPHOTO_GSDS = [2.0, 0.1]
MIN_FREE_GIB = 20.0

CRS_LV95 = "EPSG:2056"
CRS_WGS84 = "EPSG:4326"

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Exécuter ce notebook depuis la racine du dépôt projet-slow-vaud.")


In [2]:
import json
import shutil
import sys

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slowvaud.crs import wgs84_to_lv95
from slowvaud.orthophotos import build_stac_manifest_records, estimate_content_lengths, write_manifest
from slowvaud.paths import ensure_data_tree, load_config


## Configuration et centres d'agglomération

Les centres sont stockés en WGS84 dans la configuration. Les coordonnées LV95 ci-dessous servent au contrôle et aux traitements métriques.


In [3]:
cfg = load_config()
paths = ensure_data_tree()

rows = []
for agglo_key in AGGLO_KEYS:
    agglo = cfg["agglomerations"][agglo_key]
    lon = float(agglo["center_wgs84"]["lon"])
    lat = float(agglo["center_wgs84"]["lat"])
    east, north = wgs84_to_lv95(lon, lat)
    rows.append({
        "agglo_key": agglo_key,
        "label": agglo["label"],
        "lon_wgs84": lon,
        "lat_wgs84": lat,
        "east_lv95": round(east, 2),
        "north_lv95": round(north, 2),
    })

pd.DataFrame(rows)


,agglo_key,label,lon_wgs84,lat_wgs84,east_lv95,north_lv95
0,lausanne,Lausanne,6.6323,46.5197,2538124.47,1152363.14
1,berne,Berne,7.4474,46.9479,2600667.48,1199646.20
2,geneve,Genève,6.1432,46.2044,2500016.02,1117821.07
3,bale,Bâle,7.5886,47.5596,2611287.84,1267664.85
4,zurich,Zurich,8.5472,47.3769,2683719.22,1247931.48


## Contrat CRS


In [4]:
crs_rows = [
    {"objet": "agglomérations GeoAdmin", "crs": CRS_LV95, "usage": "périmètres bruts et dissous"},
    {"objet": "centres de configuration", "crs": CRS_WGS84, "usage": "points lon/lat"},
    {"objet": "OSM et GeoJSON web", "crs": CRS_WGS84, "usage": "Overpass, SITG GeoJSON"},
    {"objet": "orthophotos SWISSIMAGE STAC", "crs": CRS_LV95, "usage": "GeoTIFF/COG source"},
]
pd.DataFrame(crs_rows)


,objet,crs,usage
0,agglomérations GeoAdmin,EPSG:2056,périmètres bruts et dissous
1,centres de configuration,EPSG:4326,points lon/lat
2,OSM et GeoJSON web,EPSG:4326,"Overpass, SITG GeoJSON"
3,orthophotos SWISSIMAGE STAC,EPSG:2056,GeoTIFF/COG source


## Périmètres requis

Le manifeste orthophoto est construit à partir des périmètres VaCO dissous produits par `02_agglomerations_shapes.ipynb`.


In [5]:
boundary_rows = []
for agglo_key in AGGLO_KEYS:
    path = paths["processed"] / "agglomerations" / BOUNDARY_SOURCE / f"{agglo_key}_dissolved.geojson"
    if not path.exists():
        raise FileNotFoundError(f"Périmètre manquant: {path}")
    gdf = gpd.read_file(path)
    if str(gdf.crs) != CRS_LV95:
        raise ValueError(f"CRS inattendu pour {path}: {gdf.crs}")
    boundary_rows.append({
        "agglomeration": agglo_key,
        "features": len(gdf),
        "crs": str(gdf.crs),
        "bounds_lv95": tuple(round(value, 2) for value in gdf.total_bounds),
    })

pd.DataFrame(boundary_rows)


,agglomeration,features,crs,bounds_lv95
0,lausanne,1,EPSG:2056,"(2512518.0, 1145803.0, 2551159.0, 1177123.0)"
1,berne,1,EPSG:2056,"(2581820.0, 1181565.0, 2619062.0, 1219106.0)"
2,geneve,1,EPSG:2056,"(2485424.0, 1109578.0, 2518655.0, 1155690.0)"
3,bale,1,EPSG:2056,"(2595250.0, 1246272.0, 2639000.0, 1272264.0)"
4,zurich,1,EPSG:2056,"(2664527.0, 1222925.0, 2716149.0, 1276539.9)"


## Manifeste STAC SWISSIMAGE

La source utilisée est `ch.swisstopo.swissimage-dop10`. Le manifeste liste les GeoTIFF/COG intersectant les vrais périmètres d'agglomération, puis ajoute la taille HTTP déclarée par chaque asset.


In [6]:
records = build_stac_manifest_records(
    agglomerations=AGGLO_KEYS,
    boundary_source=BOUNDARY_SOURCE,
    gsds=ORTHOPHOTO_GSDS,
)

manifest_path = paths["manifests"] / "orthophoto_stac_manifest.csv"
write_manifest(records, manifest_path)

size_estimate = estimate_content_lengths(records)
estimate_path = paths["manifests"] / "orthophoto_stac_size_estimate.json"
estimate_path.write_text(json.dumps(size_estimate, indent=2, ensure_ascii=False), encoding="utf-8")

manifest = pd.DataFrame(records)
print(f"Manifeste: {manifest_path}")
print(f"Estimation tailles: {estimate_path}")
manifest.head()


Manifeste: /Users/marc-ed/BAS-local/repos/projet-slow-vaud/data/manifests/orthophoto_stac_manifest.csv
Estimation tailles: /Users/marc-ed/BAS-local/repos/projet-slow-vaud/data/manifests/orthophoto_stac_size_estimate.json


,agglomeration,agglomeration_label,collection,source,item_id,datetime,asset_key,gsd,proj_epsg,asset_type,checksum_multihash,href,output_path,item_bounds_lv95,intersection_area_m2
0,bale,Bâle,ch.swisstopo.swissimage-dop10,https://data.geo.admin.ch/api/stac/v0.9/collec...,swissimage-dop10_2018_2595-1252,2018-01-01T00:00:00Z,swissimage-dop10_2018_2595-1252_0.1_2056.tif,0.1,2056,image/tiff; application=geotiff; profile=cloud...,1220FA28BB666A18ADC24CB52C099DDE7137AAE1AA78EA...,https://data.geo.admin.ch/ch.swisstopo.swissim...,data/raw/orthophotos/ch.swisstopo.swissimage-d...,"[2594999.9998137904, 1251999.9997534342, 25959...",493130.54
1,bale,Bâle,ch.swisstopo.swissimage-dop10,https://data.geo.admin.ch/api/stac/v0.9/collec...,swissimage-dop10_2018_2595-1253,2018-01-01T00:00:00Z,swissimage-dop10_2018_2595-1253_0.1_2056.tif,0.1,2056,image/tiff; application=geotiff; profile=cloud...,12202A77E7E8E5A5106555419B0E32A9E944EC5C8AC38A...,https://data.geo.admin.ch/ch.swisstopo.swissim...,data/raw/orthophotos/ch.swisstopo.swissimage-d...,"[2594999.997429128, 1252999.9974860565, 259600...",145660.21
2,bale,Bâle,ch.swisstopo.swissimage-dop10,https://data.geo.admin.ch/api/stac/v0.9/collec...,swissimage-dop10_2018_2596-1251,2018-01-01T00:00:00Z,swissimage-dop10_2018_2596-1251_0.1_2056.tif,0.1,2056,image/tiff; application=geotiff; profile=cloud...,1220E44650F0A3899FE8E272AF74DEED655F999729D7B3...,https://data.geo.admin.ch/ch.swisstopo.swissim...,data/raw/orthophotos/ch.swisstopo.swissimage-d...,"[2595999.9974641134, 1250999.9958146596, 25970...",589202.98
3,bale,Bâle,ch.swisstopo.swissimage-dop10,https://data.geo.admin.ch/api/stac/v0.9/collec...,swissimage-dop10_2018_2596-1252,2018-01-01T00:00:00Z,swissimage-dop10_2018_2596-1252_0.1_2056.tif,0.1,2056,image/tiff; application=geotiff; profile=cloud...,12201C029483B1F34FF9F5C36DD356F8C438472F9EA780...,https://data.geo.admin.ch/ch.swisstopo.swissim...,data/raw/orthophotos/ch.swisstopo.swissimage-d...,"[2595999.9974641134, 1252000.0015793892, 25969...",999998.30
4,bale,Bâle,ch.swisstopo.swissimage-dop10,https://data.geo.admin.ch/api/stac/v0.9/collec...,swissimage-dop10_2018_2596-1253,2018-01-01T00:00:00Z,swissimage-dop10_2018_2596-1253_0.1_2056.tif,0.1,2056,image/tiff; application=geotiff; profile=cloud...,122087D98353965DB02442A9AD5F89B8C019782853A259...,https://data.geo.admin.ch/ch.swisstopo.swissim...,data/raw/orthophotos/ch.swisstopo.swissimage-d...,"[2595999.999295169, 1252999.9990762197, 259699...",697880.22


## Volumes par agglomération et résolution


In [7]:
asset_summary = (
    manifest.groupby(["agglomeration", "gsd"], as_index=False)
    .agg(
        assets=("href", "nunique"),
        intersection_area_m2=("intersection_area_m2", "sum"),
    )
)

size_summary = pd.DataFrame(size_estimate["summary"])
size_summary["estimated_total_gib"] = size_summary["estimated_total_bytes"] / 1024**3
asset_summary.merge(size_summary[["gsd", "sample_mean_bytes", "estimated_total_gib"]], on="gsd", how="left")


,agglomeration,gsd,assets,intersection_area_m2,sample_mean_bytes,estimated_total_gib
0,bale,0.1,2305,1.953732e+09,67940000,743.913072
1,bale,2.0,2305,1.953732e+09,169053,1.851051
2,berne,0.1,2122,1.760352e+09,67940000,743.913072
3,berne,2.0,2122,1.760352e+09,169053,1.851051
4,geneve,0.1,1358,1.109787e+09,67940000,743.913072
5,geneve,2.0,1358,1.109787e+09,169053,1.851051
6,lausanne,0.1,1306,9.672249e+08,67940000,743.913072
7,lausanne,2.0,1306,9.672249e+08,169053,1.851051
8,zurich,0.1,4678,4.041205e+09,67940000,743.913072
9,zurich,2.0,4678,4.041205e+09,169053,1.851051


## Contrôle espace disque


In [8]:
free_bytes = shutil.disk_usage(PROJECT_ROOT).free
required_bytes = int(sum(row["estimated_total_bytes"] for row in size_estimate["summary"]))
reserve_bytes = int(MIN_FREE_GIB * 1024**3)

disk_rows = [{
    "free_gib": round(free_bytes / 1024**3, 1),
    "estimated_required_gib": round(required_bytes / 1024**3, 1),
    "reserve_gib": MIN_FREE_GIB,
    "download_all_fits": required_bytes <= max(free_bytes - reserve_bytes, 0),
}]
pd.DataFrame(disk_rows)


,free_gib,estimated_required_gib,reserve_gib,download_all_fits
0,68.7,745.8,20.0,False
